# Confusion Matrix Report

**Purpose**: Detailed confusion matrix analysis with per-digit metrics

**Outputs**: Heatmap, per-class metrics, confusion analysis

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import yaml

sys.path.insert(0, '.')

from src.models.factory import ModelFactory
from src.data.dataloader import create_test_dataloader
from sklearn.metrics import confusion_matrix, classification_report

# Setup
config_path = 'configs/experiments/baseline.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model
model = ModelFactory.create(
    name=config['model']['name'],
    num_classes=config['model']['num_classes'],
    hidden_features=config['model']['hidden_features']
)

checkpoint_path = Path('artifacts/checkpoints/last.pt')
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Model loaded")
else:
    print("⚠️  No checkpoint found")

model.to(device)
model.eval()

# Data
test_loader = create_test_dataloader(
    data_dir=config['data']['data_dir'],
    batch_size=64
)

# Predictions
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(f"✓ Predictions generated: {len(all_preds)} samples")

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Normalize for percentage
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Count heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10),
            cbar_kws={'label': 'Count'}, ax=ax1, square=True)
ax1.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax1.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax1.set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')

# Percentage heatmap
sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='RdYlGn', 
            xticklabels=range(10), yticklabels=range(10),
            cbar_kws={'label': 'Percentage (%)'}, ax=ax2, square=True, vmin=0, vmax=100)
ax2.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax2.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax2.set_title('Confusion Matrix (Percentages)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('notebooks/assets/figures/report_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix heatmap saved")

In [ ]:
# Classification report
report = classification_report(all_labels, all_preds, output_dict=True)

per_class = pd.DataFrame({
    'Digit': range(10),
    'Precision': [report[str(i)]['precision'] for i in range(10)],
    'Recall': [report[str(i)]['recall'] for i in range(10)],
    'F1-Score': [report[str(i)]['f1-score'] for i in range(10)],
    'Support': [int(report[str(i)]['support']) for i in range(10)]
})

print("\n" + "="*70)
print("📊 Per-Class Performance Metrics")
print("="*70)
print(per_class.to_string(index=False))
print("="*70)

print(f"\nMacro Avg: P={report['macro avg']['precision']:.4f}, R={report['macro avg']['recall']:.4f}, F1={report['macro avg']['f1-score']:.4f}")
print(f"Weighted Avg: P={report['weighted avg']['precision']:.4f}, R={report['weighted avg']['recall']:.4f}, F1={report['weighted avg']['f1-score']:.4f}")

## 5. Save Report